# 🤖 Crypto ML Predictor

Machine Learning untuk prediksi harga BTC 7 hari ke depan.

**Cara pakai:** Run semua cell dari atas ke bawah. Model otomatis ter-train dan prediksi dikirim ke Telegram.

**Author:** sadz-hubz

## Cell 1 — Install & Import

In [ ]:
!pip install xgboost scikit-learn yfinance requests joblib -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import json
import warnings
from datetime import datetime, timedelta
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import joblib

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

print("✅ Libraries loaded!")
print(f"⏰ {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Cell 2 — Ambil Data

In [ ]:
# ============================================================
# AMBIL DATA DARI API
# ============================================================

print("📡 Fetching data...\n")

# --- BTC Price dari CoinGecko ---
url = "https://api.coingecko.com/api/v3/coins/bitcoin/market_chart"
params = {'vs_currency': 'usd', 'days': 730, 'interval': 'daily'}
response = requests.get(url, params=params)
data = response.json()

df = pd.DataFrame(data['prices'], columns=['timestamp', 'price'])
df['date'] = pd.to_datetime(df['timestamp'], unit='ms')
vol = pd.DataFrame(data['total_volumes'], columns=['timestamp', 'volume'])
df['volume'] = vol['volume']
df = df.set_index('date').drop('timestamp', axis=1)
print(f"✅ BTC: {len(df)} data points | {df.index[0].date()} → {df.index[-1].date()}")

# --- Fear & Greed Index ---
try:
    fng_url = "https://api.alternative.me/fng/?limit=730"
    fng_data = requests.get(fng_url).json()['data']
    fng_df = pd.DataFrame(fng_data)
    fng_df['date'] = pd.to_datetime(fng_df['timestamp'], unit='s')
    fng_df['fng'] = fng_df['value'].astype(float)
    fng_df = fng_df.set_index('date').sort_index().resample('D').ffill()
    df = df.join(fng_df[['fng']], how='left')
    print(f"✅ Fear & Greed: loaded")
except:
    df['fng'] = 50
    print(f"⚠️ F&G failed, using default 50")

# --- DXY & S&P500 ---
try:
    import yfinance as yf
    dxy = yf.download('DX-Y.NYB', period='2y', progress=False)['Close']
    sp = yf.download('^GSPC', period='2y', progress=False)['Close']
    df = df.join(dxy.rename('dxy'), how='left')
    df = df.join(sp.rename('sp500'), how='left')
    print(f"✅ DXY & S&P500: loaded")
except:
    print(f"⚠️ Macro data failed, skipping")

df = df.ffill().bfill()
print(f"\n✅ Dataset: {df.shape[0]} rows × {df.shape[1]} columns")
df.tail()

## Cell 3 — Hitung Indikator Teknikal

In [ ]:
# ============================================================
# FEATURE ENGINEERING
# ============================================================

# Returns
for d in [1, 3, 7, 14, 30]:
    df[f'ret_{d}d'] = df['price'].pct_change(d)
df['log_ret_1d'] = np.log(df['price'] / df['price'].shift(1))

# SMA
for period in [7, 14, 20, 50, 100, 200]:
    df[f'sma_{period}'] = df['price'].rolling(period).mean()
    df[f'price_vs_sma_{period}'] = (df['price'] - df[f'sma_{period}']) / df[f'sma_{period}']

# EMA & MACD
df['ema_12'] = df['price'].ewm(span=12, adjust=False).mean()
df['ema_26'] = df['price'].ewm(span=26, adjust=False).mean()
df['macd'] = df['ema_12'] - df['ema_26']
df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()
df['macd_hist'] = df['macd'] - df['macd_signal']

# RSI
def calc_rsi(series, period):
    delta = series.diff()
    gain = delta.where(delta > 0, 0).rolling(period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(period).mean()
    rs = gain / loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

df['rsi_14'] = calc_rsi(df['price'], 14)
df['rsi_7'] = calc_rsi(df['price'], 7)

# Bollinger Bands
df['bb_mid'] = df['price'].rolling(20).mean()
bb_std = df['price'].rolling(20).std()
df['bb_upper'] = df['bb_mid'] + 2 * bb_std
df['bb_lower'] = df['bb_mid'] - 2 * bb_std
df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / df['bb_mid']
df['bb_position'] = (df['price'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'])

# ATR
high_low = df['price'].rolling(2).max() - df['price'].rolling(2).min()
df['atr_7'] = high_low.rolling(7).mean()
df['atr_14'] = high_low.rolling(14).mean()

# Volume
df['vol_sma_7'] = df['volume'].rolling(7).mean()
df['vol_sma_30'] = df['volume'].rolling(30).mean()
df['vol_ratio_7'] = df['volume'] / df['vol_sma_7']
df['vol_ratio_30'] = df['volume'] / df['vol_sma_30']

# Volatility
df['volatility_7d'] = df['ret_1d'].rolling(7).std()
df['volatility_30d'] = df['ret_1d'].rolling(30).std()

# Momentum
for d in [7, 14, 30]:
    df[f'momentum_{d}d'] = df['price'] / df['price'].shift(d) - 1

# Macro features
if 'dxy' in df.columns:
    df['dxy_ret_1d'] = df['dxy'].pct_change(1)
    df['dxy_ret_7d'] = df['dxy'].pct_change(7)
if 'sp500' in df.columns:
    df['sp500_ret_1d'] = df['sp500'].pct_change(1)
    df['sp500_ret_7d'] = df['sp500'].pct_change(7)

# Calendar
df['day_of_week'] = df.index.dayofweek
df['month'] = df.index.month
df['is_weekend'] = (df.index.dayofweek >= 5).astype(int)

print(f"✅ Features added: {len(df.columns)} total columns")

## Cell 4 — Buat Target & Bersihkan Data

In [ ]:
# ============================================================
# TARGET VARIABLE
# ============================================================

# Target: 7 hari ke depan naik (1) atau turun (0)
df['target_ret_7d'] = df['price'].shift(-7) / df['price'] - 1
df['target_dir_7d'] = (df['target_ret_7d'] > 0).astype(int)

# Drop NaN
df_clean = df.dropna()

print(f"📊 Clean samples: {len(df_clean)}")
print(f"📊 Date range: {df_clean.index[0].date()} → {df_clean.index[-1].date()}")
print(f"\n🎯 Target balance:")
print(df_clean['target_dir_7d'].value_counts(normalize=True).round(3))

## Cell 5 — Visualisasi Data

In [ ]:
# ============================================================
# VISUALISASI
# ============================================================

fig, axes = plt.subplots(3, 2, figsize=(16, 15))

# Price + SMA
ax = axes[0, 0]
ax.plot(df.index, df['price'], label='BTC', linewidth=1.5)
for p in [20, 50, 200]:
    ax.plot(df.index, df[f'sma_{p}'], label=f'SMA {p}', alpha=0.7)
ax.set_title('BTC Price + Moving Averages')
ax.legend()
ax.set_ylabel('Price (USD)')

# RSI
ax = axes[0, 1]
ax.plot(df.index, df['rsi_14'], color='purple', linewidth=1)
ax.axhline(y=70, color='red', linestyle='--', alpha=0.5)
ax.axhline(y=30, color='green', linestyle='--', alpha=0.5)
ax.fill_between(df.index, 30, 70, alpha=0.1, color='gray')
ax.set_title('RSI 14')
ax.set_ylim(0, 100)

# MACD
ax = axes[1, 0]
ax.plot(df.index, df['macd'], label='MACD', color='blue')
ax.plot(df.index, df['macd_signal'], label='Signal', color='red')
ax.bar(df.index, df['macd_hist'], alpha=0.3, color='gray')
ax.set_title('MACD')
ax.legend()

# Bollinger Bands
ax = axes[1, 1]
ax.plot(df.index, df['price'], linewidth=1)
ax.plot(df.index, df['bb_upper'], '--', alpha=0.5)
ax.plot(df.index, df['bb_lower'], '--', alpha=0.5)
ax.fill_between(df.index, df['bb_lower'], df['bb_upper'], alpha=0.1)
ax.set_title('Bollinger Bands')

# Fear & Greed
ax = axes[2, 0]
ax.plot(df.index, df['fng'], color='orange', linewidth=1)
ax.axhline(y=25, color='red', linestyle='--', alpha=0.5)
ax.axhline(y=75, color='green', linestyle='--', alpha=0.5)
ax.set_title('Fear & Greed Index')
ax.set_ylim(0, 100)

# Returns distribution
ax = axes[2, 1]
ax.hist(df['ret_1d'].dropna(), bins=80, alpha=0.7, edgecolor='black')
ax.axvline(x=0, color='red', linestyle='--')
ax.set_title('Daily Returns Distribution')

plt.tight_layout()
plt.show()
print("✅ Visualization done!")

## Cell 6 — Training Model

In [ ]:
# ============================================================
# TRAIN ML MODELS
# ============================================================

# Pilih features
exclude = ['price', 'volume',
           'sma_7', 'sma_14', 'sma_20', 'sma_50', 'sma_100', 'sma_200',
           'ema_12', 'ema_26', 'bb_mid', 'bb_upper', 'bb_lower',
           'vol_sma_7', 'vol_sma_30', 'atr_7', 'atr_14',
           'dxy', 'sp500', 'target_ret_7d', 'target_dir_7d']

feature_cols = [c for c in df_clean.columns if c not in exclude]

X = df_clean[feature_cols]
y = df_clean['target_dir_7d']

print(f"📊 Features: {len(feature_cols)}")
print(f"📊 Samples: {len(X)}")

# Walk-forward validation
def evaluate(model, X, y, n_splits=5):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    results = []
    all_preds, all_actuals = [], []
    
    for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
        
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_te_s = scaler.transform(X_te)
        
        model.fit(X_tr_s, y_tr)
        preds = model.predict(X_te_s)
        acc = accuracy_score(y_te, preds)
        
        results.append(acc)
        all_preds.extend(preds)
        all_actuals.extend(y_te)
        print(f"  Fold {fold+1}: {acc:.4f}")
    
    return np.mean(results), np.array(all_preds), np.array(all_actuals)

# Bandingkan model
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, random_state=42),
    'XGBoost': xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42, use_label_encoder=False, eval_metric='logloss')
}

print("\n" + "=" * 50)
print("MODEL COMPARISON")
print("=" * 50)

best_acc = 0
best_name = ""
best_model = None

for name, model in models.items():
    print(f"\n🔹 {name}")
    acc, _, _ = evaluate(model, X, y)
    print(f"  → Avg Accuracy: {acc:.4f}")
    if acc > best_acc:
        best_acc = acc
        best_name = name
        best_model = model

print(f"\n{'=' * 50}")
print(f"🏆 BEST: {best_name} | Accuracy: {best_acc:.4f}")
print(f"{'=' * 50}")

## Cell 7 — Evaluasi Detail

In [ ]:
# ============================================================
# EVALUASI DETAIL — Best Model
# ============================================================

# Retrain dengan best model
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
best_model.fit(X_scaled, y)

# Walk-forward evaluation
acc, preds, actuals = evaluate(best_model, X, y)

print("\n" + "=" * 50)
print(f"EVALUASI — {best_name}")
print("=" * 50)
print(classification_report(actuals, preds, target_names=['TURUN 📉', 'NAIK 📈']))

# Confusion Matrix
cm = confusion_matrix(actuals, preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['TURUN', 'NAIK'], yticklabels=['TURUN', 'NAIK'])
plt.title(f'Confusion Matrix — {best_name}')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

# Feature Importance
if hasattr(best_model, 'feature_importances_'):
    importance = pd.DataFrame({
        'feature': feature_cols,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    plt.figure(figsize=(12, 8))
    sns.barplot(data=importance.head(20), x='importance', y='feature', palette='viridis')
    plt.title(f'Top 20 Feature Importance — {best_name}')
    plt.tight_layout()
    plt.show()
    
    print("\n📊 TOP 10 FEATURES:")
    for _, row in importance.head(10).iterrows():
        print(f"  {row['feature']:30s} {row['importance']:.4f}")

## Cell 8 — Prediksi Hari Ini

In [ ]:
# ============================================================
# PREDIKSI HARI INI
# ============================================================

# Ambil data terbaru
latest = df[feature_cols].iloc[-1:]
missing = set(feature_cols) - set(latest.columns)
for m in missing:
    latest[m] = 0
latest = latest[feature_cols]

# Predict
latest_scaled = scaler.transform(latest)
pred = best_model.predict(latest_scaled)[0]
proba = best_model.predict_proba(latest_scaled)[0]

btc_price = df['price'].iloc[-1]
direction = "🟢 NAIK" if pred == 1 else "🔴 TURUN"
confidence = max(proba) * 100

# Info tambahan
rsi = df['rsi_14'].iloc[-1]
fng = df['fng'].iloc[-1]
macd_val = df['macd_hist'].iloc[-1]
bb_pos = df['bb_position'].iloc[-1]

# Smart DCA
threshold = 77000
dca = 100000 if btc_price < threshold else 50000
dca_mode = "agresif 🔥" if btc_price < threshold else "pelan 🐢"

emoji_fng = "😱" if fng < 25 else "😰" if fng < 50 else "😐" if fng < 75 else "🤑"

print("╔══════════════════════════════════╗")
print("║  📊 BTC PREDICTION (7 DAYS)     ║")
print("╠══════════════════════════════════╣")
print(f"║  💰 BTC: ${btc_price:>12,.2f}      ║")
print(f"║  📈 Arah: {direction:<20s}  ║")
print(f"║  🎯 Conf: {confidence:>5.1f}%              ║")
print("╠══════════════════════════════════╣")
print(f"║  RSI: {rsi:.1f} | MACD: {'Bull' if macd_val > 0 else 'Bear':>5s}      ║")
print(f"║  BB%: {bb_pos:.1f}% | F&G: {fng:.0f} {emoji_fng}        ║")
print("╠══════════════════════════════════╣")
print(f"║  💰 DCA: Rp {dca:,} ({dca_mode})  ║")
print(f"║  NAIK: {proba[1]*100:.1f}% | TURUN: {proba[0]*100:.1f}%      ║")
print("╚══════════════════════════════════╝")
print(f"\n🤖 Model: {best_name}")
print(f"⏰ {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Cell 9 — Save Model & Download

In [ ]:
# ============================================================
# SAVE MODEL
# ============================================================

import pickle

joblib.dump(best_model, 'crypto_model.pkl')
joblib.dump(scaler, 'crypto_scaler.pkl')
with open('feature_cols.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

print("✅ Model saved!")
print("  - crypto_model.pkl")
print("  - crypto_scaler.pkl")
print("  - feature_cols.pkl")

# Download
from google.colab import files
files.download('crypto_model.pkl')
files.download('crypto_scaler.pkl')
files.download('feature_cols.pkl')

print("\n✅ Download complete!")
print("\n📋 NEXT STEPS:")
print("  1. Upload 3 file di atas ke Termux (~/.hermes/)")
print("  2. Setup Telegram bot token di crypto_predict.py")
print("  3. Setup cron job: 0 9 * * * python3 ~/.hermes/crypto_predict.py")

## Cell 10 — Kirim ke Telegram (Opsional)

In [ ]:
# ============================================================
# KIRIM PREDIKSI KE TELEGRAM
# ============================================================
# Setup:
# 1. Buka Telegram → @BotFather → /newbot → copy TOKEN
# 2. Buka bot lo → kirim /start
# 3. Buka: https://api.telegram.org/bot<TOKEN>/getUpdates
# 4. Copy chat.id

TELEGRAM_TOKEN = "PASTE_TOKEN_LO_DISINI"
TELEGRAM_CHAT_ID = "PASTE_CHAT_ID_LO_DISINI"

def send_telegram(message):
    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
    payload = {'chat_id': TELEGRAM_CHAT_ID, 'text': message, 'parse_mode': 'HTML'}
    response = requests.post(url, json=payload)
    if response.status_code == 200:
        print("✅ Telegram sent!")
    else:
        print(f"⚠️ Failed: {response.text}")

message = f"""
╔══════════════════════════════════╗
║  📊 <b>BTC DAILY SIGNAL</b>          ║
╠══════════════════════════════════╣
║  💰 BTC: <b>${btc_price:,.2f}</b>
║  📈 Prediksi 7d: <b>{direction}</b>
║  🎯 Confidence: <b>{confidence:.1f}%</b>
╠══════════════════════════════════╣
║  TEKNIKAL:
║  RSI 14: {rsi:.1f}
║  MACD: {'Bullish 📈' if macd_val > 0 else 'Bearish 📉'}
║  BB Position: {bb_pos:.1f}%
║  F&G: {fng:.0f} {emoji_fng}
╠══════════════════════════════════╣
║  💰 SMART DCA: <b>Rp {dca:,}</b>
║  Mode: {dca_mode}
╠══════════════════════════════════╣
║  NAIK: {proba[1]*100:.1f}% | TURUN: {proba[0]*100:.1f}%
╚══════════════════════════════════╝

⏰ {datetime.now().strftime('%Y-%m-%d %H:%M')} WIB
🤖 Model: {best_name}
⚠️ Bukan saran finansial. DYOR!
"""

# Uncomment untuk kirim:
# send_telegram(message)

print("📋 Pesan Telegram:")
print(message)
print("\n💡 Uncomment send_telegram(message) di atas untuk kirim otomatis")